In [ ]:
import re
import random
import numpy as np
import torch
import pandas as pd
SEED = 39
def set_seed(seed = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed()
dataset = pd.read_table("data/dataset_small.tsv", quoting=3, names=["ru_id", "russian", "en_id", "english"])
dataset.head()

In [11]:
print(dataset.shape)
print(dataset.isna().sum())

(824903, 4)
ru_id      0
russian    0
en_id      0
english    0
dtype: int64


In [ ]:
data = dataset[["russian", "english"]]
data.head()

,russian,english
0,Один раз в жизни я делаю хорошее дело... И оно...,For once in my life I'm doing a good deed... A...
1,Давайте что-нибудь попробуем!,Let's try something.
2,Мне пора идти спать.,I have to go to sleep.
3,Что ты делаешь?,What are you doing?
4,Что ты делаешь?,What do you make?


In [ ]:
from src.preprocessing import preprocess_parallel_data
data, _ = preprocess_parallel_data(dataset)

In [16]:
data

,russian,english
0,Один раз в жизни я делаю хорошее дело... И оно...,For once in my life I'm doing a good deed... A...
1,Давайте что-нибудь попробуем!,Let's try something.
2,Мне пора идти спать.,I have to go to sleep.
3,Что ты делаешь?,What are you doing?
4,Что ты делаешь?,What do you make?
...,...,...
824069,Что будешь готовить на обед?,What are you going to make for lunch?
824070,Что вы должны приготовить на обед?,What do you have to make for lunch?
824071,Что ты должен приготовить на обед?,What do you have to make for lunch?
824072,А у вас ещё есть?,Do you have any more?


In [17]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(data, train_size = 0.8, random_state = SEED, shuffle = True)
valid, test = train_test_split(temp, train_size = 0.5, random_state = SEED, shuffle = True)
print(f"train: {train.shape}, valid: {valid.shape}, test: {test.shape}")

train: (659259, 2), valid: (82407, 2), test: (82408, 2)


In [18]:
from collections import Counter

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text)
train_data = train.copy()
valid_data = valid.copy()
test_data = test.copy()
for split in [train_data, valid_data, test_data]:
    split["ru_tokens"] = split["russian"].apply(tokenize)
    split["en_tokens"] = split["english"].apply(tokenize)

SPECIAL_TOKENS = ["<pad>", "<unk>", "<sos>", "<eos>"]
def build_vocab(token_lists, min_freq = 2):
    counter = Counter(
        token for tokens in token_lists for token in tokens
    )
    vocab = {token : idx for idx, token in enumerate(SPECIAL_TOKENS)}
    for token, frequency in counter.items():
        if frequency >= min_freq and token not in vocab:
            vocab[token] = len(vocab)
    return vocab
ru_vocab = build_vocab(train_data["ru_tokens"])
en_vocab = build_vocab(train_data["en_tokens"])
print("Размер русского словаря:", len(ru_vocab))
print("Размер английского словаря:", len(en_vocab))

Размер русского словаря: 58739
Размер английского словаря: 22890


In [19]:
PAD_IDX = en_vocab["<pad>"]
UNK_RU_IDX = ru_vocab["<unk>"]
UNK_EN_IDX = en_vocab["<unk>"]

def tokens_to_ids(tokens, vocab, add_sos_eos=False):
    ids = [vocab.get(token, vocab["<unk>"]) for token in tokens]

    if add_sos_eos:
        ids = [vocab["<sos>"]] + ids + [vocab["<eos>"]]

    return ids

for split in [train_data, valid_data, test_data]:
    split["ru_ids"] = split["ru_tokens"].apply(
        lambda tokens: tokens_to_ids(tokens, ru_vocab)
    )

    split["en_ids"] = split["en_tokens"].apply(
        lambda tokens: tokens_to_ids(tokens, en_vocab, add_sos_eos=True)
    )

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader
from src.dataset import (
    TranslationDataset,
    TranslationCollator,
)
RU_PAD_IDX = ru_vocab["<pad>"]
EN_PAD_IDX = en_vocab["<pad>"]
BATCH_SIZE = 64
train_dataset = TranslationDataset(train_data)
valid_dataset = TranslationDataset(valid_data)
test_dataset = TranslationDataset(test_data)

collator = TranslationCollator(
    ru_pad_idx=RU_PAD_IDX,
    en_pad_idx=EN_PAD_IDX
)

generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=0,
    generator=generator
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0
)


In [ ]:
from src.model_baseline import Encoder, Decoder, Seq2Seq
EMBEDDING_DIM = 256
HIDDEN_DIM = 256
NUM_LAYERS = 1
DROPOUT = 0.2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = Encoder(
    input_dim=len(ru_vocab),
    embed_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    pad_idx=RU_PAD_IDX,
    dropout=DROPOUT
)

decoder = Decoder(
    output_dim=len(en_vocab),
    embed_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    pad_idx=EN_PAD_IDX,
    dropout=DROPOUT
)

model = Seq2Seq(encoder, decoder).to(DEVICE)

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=EN_PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
import copy
import time
from src.train import train_epoch, validate_epoch
NUM_EPOCHS = 10

train_losses = []
val_losses = []

best_val_loss = float("inf")
best_model_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()

    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        DEVICE,
        target_pad_idx=EN_PAD_IDX
    )

    val_loss = validate_epoch(
        model,
        valid_loader,
        criterion,
        DEVICE,
        target_pad_idx=EN_PAD_IDX,
        teacher_forcing_rate=0.0
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())

    epoch_time = time.time() - start_time

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train loss: {train_loss:.4f} | "
        f"val loss: {val_loss:.4f} | "
        f"time: {epoch_time:.1f}s"
    )

In [31]:
model.load_state_dict(best_model_state)

<All keys matched successfully>

In [ ]:
from pathlib import Path

Path("models").mkdir(exist_ok=True)

checkpoint = {
    "model_state_dict": best_model_state,
    "ru_vocab": ru_vocab,
    "en_vocab": en_vocab,
    "embedding_dim": EMBEDDING_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS
}

torch.save(checkpoint, "models/best_seq2seq_baseline.pt")

In [33]:
import re
import torch
from tqdm.auto import tqdm

model.load_state_dict(best_model_state)
model.to(DEVICE)
model.eval()

id_to_en_token = {
    token_id: token
    for token, token_id in en_vocab.items()
}

In [34]:
def detokenize_english(tokens):
    text = " ".join(tokens)
    text = re.sub(r"\s+([.,!?;:])", r"\1", text)
    text = re.sub(r"\s*'\s*", "'", text)
    text = re.sub(r"\s*-\s*", "-", text)

    return text

In [35]:
def translate_sentence(
    sentence,
    model,
    ru_vocab,
    en_vocab,
    max_length=50
):
    model.eval()

    source_ids = [
        ru_vocab.get(token, ru_vocab["<unk>"])
        for token in tokenize(sentence)
    ]

    source = torch.tensor(
        [source_ids],
        dtype=torch.long,
        device=DEVICE
    )

    source_lengths = torch.tensor(
        [len(source_ids)],
        dtype=torch.long
    )

    generated_ids = []

    with torch.no_grad():
        _, hidden, cell = model.encoder(
            source,
            source_lengths
        )

        input_token = torch.tensor(
            [en_vocab["<sos>"]],
            dtype=torch.long,
            device=DEVICE
        )

        for _ in range(max_length):
            prediction, hidden, cell = model.decoder(
                input_token,
                hidden,
                cell
            )

            next_token_id = prediction.argmax(dim=1).item()

            if next_token_id == en_vocab["<eos>"]:
                break

            generated_ids.append(next_token_id)

            input_token = torch.tensor(
                [next_token_id],
                dtype=torch.long,
                device=DEVICE
            )

    generated_tokens = [
        id_to_en_token[token_id]
        for token_id in generated_ids
    ]

    return detokenize_english(generated_tokens)

In [36]:
for _, row in valid_data.sample(5, random_state=SEED).iterrows():
    prediction = translate_sentence(
        row["russian"],
        model,
        ru_vocab,
        en_vocab
    )

    print("RU:       ", row["russian"])
    print("REFERENCE:", row["english"])
    print("PREDICT:  ", prediction)
    print()

RU:        Не заставляйте меня это есть, пожалуйста.
REFERENCE: Please don't make me eat that.
PREDICT:   Please don't make me eat it.

RU:        Она обещала мне позвонить.
REFERENCE: She promised to give me a phone call.
PREDICT:   She promised me to call her.

RU:        Том сказал, что не хочет с нами идти.
REFERENCE: Tom said that he didn't want to go with us.
PREDICT:   Tom said that he didn't want to go with us.

RU:        Я начинаю уставать от твоих жалоб.
REFERENCE: I'm starting to get tired of your complaining.
PREDICT:   I'm starting to get your complaints.

RU:        Мы хотим увидеть результаты.
REFERENCE: We want to see results.
PREDICT:   We want to see results.



In [38]:
import sacrebleu

predictions = [
    translate_sentence(
        russian,
        model,
        ru_vocab,
        en_vocab
    )
    for russian in tqdm(valid_data["russian"])
]

references = valid_data["english"].tolist()

bleu = sacrebleu.corpus_bleu(
    predictions,
    [references]
)

chrf = sacrebleu.corpus_chrf(
    predictions,
    [references]
)

print(f"Validation BLEU: {bleu.score:.2f}")
print(f"Validation chrF: {chrf.score:.2f}")

  0%|          | 0/82407 [00:00<?, ?it/s]

Validation BLEU: 44.29
Validation chrF: 59.03
